In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [3]:
dataset = pd.read_csv("CKD.csv")
dataset

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [5]:
indep1 = dataset.drop(columns=['classification'])

In [6]:
dep = dataset['classification']

In [8]:
indep = pd.get_dummies(indep1,drop_first = True)

In [9]:
dataset['classification'].value_counts()

classification
yes    249
no     150
Name: count, dtype: int64

In [10]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split (indep, dep, test_size =1/3,random_state =0)

In [11]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [13]:
from sklearn.tree import DecisionTreeClassifier

In [15]:
from sklearn.model_selection import GridSearchCV
param_grid = {'criterion':['entropy', 'gini'],
              'max_features':['sqrt', 'log2', None],
              'splitter' : ['best', 'random']}

grid = GridSearchCV(DecisionTreeClassifier(), param_grid, refit=True , verbose =3, n_jobs=-1, scoring = 'f1_weighted')

grid.fit(x_train,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


,estimator,DecisionTreeClassifier()
,param_grid,"{'criterion': ['entropy', 'gini'], 'max_features': ['sqrt', 'log2', ...], 'splitter': ['best', 'random']}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [16]:
re = grid.cv_results_
grid_prediction = grid.predict(x_test)

In [17]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_prediction)

In [18]:
from sklearn.metrics import classification_report 
clf_report = classification_report(y_test, grid_prediction)

In [19]:
print (cm)
print (clf_report)

[[48  3]
 [ 3 79]]
              precision    recall  f1-score   support

          no       0.94      0.94      0.94        51
         yes       0.96      0.96      0.96        82

    accuracy                           0.95       133
   macro avg       0.95      0.95      0.95       133
weighted avg       0.95      0.95      0.95       133



In [20]:
from sklearn.metrics import f1_score
f1_macro = f1_score (y_test, grid_prediction, average = 'weighted')
print (" the f1_macro value for best parameter {}:". format(grid.best_params_), f1_macro)

 the f1_macro value for best parameter {'criterion': 'entropy', 'max_features': 'log2', 'splitter': 'random'}: 0.9548872180451128


In [21]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, grid.predict_proba(x_test)[:,1])

0.9522955523672884

In [25]:
table = pd.DataFrame.from_dict(re)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.011165,0.005699,0.011341,0.002243,entropy,sqrt,best,"{'criterion': 'entropy', 'max_features': 'sqrt...",0.944707,0.885265,0.888515,0.922492,1.000000,0.928196,0.042139,12
1,0.006207,0.000947,0.011457,0.001731,entropy,sqrt,random,"{'criterion': 'entropy', 'max_features': 'sqrt...",1.000000,0.981014,0.962573,0.961826,0.981217,0.977326,0.014147,2
2,0.006360,0.001269,0.012778,0.001980,entropy,log2,best,"{'criterion': 'entropy', 'max_features': 'log2...",0.926567,0.962264,0.981217,0.942332,0.962264,0.954929,0.018771,8
3,0.004878,0.000804,0.013119,0.001556,entropy,log2,random,"{'criterion': 'entropy', 'max_features': 'log2...",1.000000,0.981014,0.981217,0.962573,0.962264,0.977414,0.014052,1
4,0.007097,0.000781,0.012973,0.000864,entropy,None,best,"{'criterion': 'entropy', 'max_features': None,...",0.872428,0.942166,0.962573,0.942332,0.981031,0.940106,0.036794,11
5,0.004814,0.002018,0.010449,0.001878,entropy,None,random,"{'criterion': 'entropy', 'max_features': None,...",1.000000,0.943041,0.944023,0.962264,1.000000,0.969866,0.025539,5
6,0.003348,0.000411,0.008956,0.001518,gini,sqrt,best,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.962963,0.981014,0.981217,0.981217,0.943093,0.969901,0.015142,4
7,0.002818,0.000217,0.008421,0.000845,gini,sqrt,random,"{'criterion': 'gini', 'max_features': 'sqrt', ...",0.890467,1.000000,0.962573,0.981217,0.962264,0.959304,0.037133,7
8,0.003880,0.000548,0.012008,0.001491,gini,log2,best,"{'criterion': 'gini', 'max_features': 'log2', ...",0.981569,0.943041,0.944023,0.962264,0.923652,0.950910,0.019602,9
9,0.004538,0.000990,0.007706,0.001698,gini,log2,random,"{'criterion': 'gini', 'max_features': 'log2', ...",0.962963,0.942166,0.962573,0.962264,0.981217,0.962237,0.012359,6
